<!-- REFACTOR-COMMENT: POST_PROCESSING -->
### Refactor Comment (Post-Processing)
- **Purpose**: Generate stock-focused post-processing tables and figures.
- **Primary inputs**: runs/<run_name>/{Reference,Ban}/stock.csv
- **Primary outputs**: stock_heater_* csv exports and stock figures
- **Refactor role**: Keep as specialized reporting notebook; align naming and output directory conventions.
- **Data policy**: keep existing simulation data in place during refactor; no run deletion in migration phases.


<!-- NOTEBOOK-WORKFLOW -->
## Notebook Workflow
### What This Notebook Does
1. Load stock snapshots (Reference/Ban or configured scenarios).
2. Build grouped stock tables by occupancy/income/heating/performance.
3. Export stock transition CSVs and corresponding charts.

### Inputs
- `runs/<run_name>/Reference/stock.csv`
- `runs/<run_name>/Ban/stock.csv` (for comparison sections)

### Outputs
- `landlord_income.csv`
- `stock_heater_ini.csv`
- `stock_heater_end.csv`
- `stock_heater_diff_value.csv`
- `stock_heater_diff_ratio.csv`
- `stock_heater_end_reference.csv`

### How To Reuse
- Set `name` to a run folder under `reporting/runs/` containing `stock.csv` outputs.


In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
import sys

# Add the project directory to the path
sys.path.append(os.path.join(os.getcwd(), '..', '..', '..', '..'))

from project.utils import make_stacked_bar_subplot
from project.input.resources import resources_data





In [ ]:
name = 'test'
# Refactor: reporting run data is under reporting/runs
folder = os.path.join('runs', name)

## Stock

In [ ]:
stock = pd.read_csv(os.path.join(folder, 'Reference', 'stock.csv'))
stock.set_index(['Occupancy status', 'Income owner', 'Income tenant', 'Housing type', 'Heating system', 'Performance'], inplace=True)
stock_ini, stock_end = stock.iloc[:, 0], stock.iloc[:, -1]

In [ ]:
from project.utils import make_stacked_bar_subplot

In [ ]:
stock_ini

In [ ]:
from matplotlib.colors import to_rgb

def text_color_based_on_background(color):
    # Convert the background color to RGB format
    rgb = to_rgb(color)
    # Calculate the luminance of the color
    luminance = 0.299 * rgb[0] + 0.587 * rgb[1] + 0.114 * rgb[2]
    # Return 'white' for dark backgrounds and 'black' for light backgrounds
    return 'white' if luminance < 0.5 else 'black'

In [ ]:
ds = stock_ini.xs('Privately rented', level='Occupancy status').groupby('Income owner').sum()
ds.to_csv(os.path.join(folder, 'landlord_income.csv'))

colors = [resources_data['colors'][c] for c in ds.index]

# Plotting the pie chart
fig, ax = plt.subplots(figsize=(12.8, 9.6))
wedges, texts, autotexts = ax.pie(ds, colors=colors, autopct='%1.0f%%', startangle=90, pctdistance=0.85, labels=ds.index)

for autotext, color in zip(autotexts, colors):
    autotext.set_color(text_color_based_on_background(color))  # Set text color based on slice color
    autotext.set_fontsize(18)  # Set the fontsize

# Draw a circle at the center of pie to make it look like a donut
centre_circle = plt.Circle((0, 0), 0.70, fc='white')
fig = plt.gcf()
fig.gca().add_artist(centre_circle)

# Adjusting the plot to mimic the previous formatting
plt.title('Distribution of income group among landlords', fontsize=20)
ax.set_ylabel('')  # Removing y-label
plt.tight_layout()

plt.savefig(os.path.join(folder, 'landlord_income.png'))

In [ ]:
subplot_groups= ['Housing type', 'Occupancy status'] 
index_group = 'Income tenant'
stack_group ='Heating system'

In [ ]:
df = stock_ini.groupby(subplot_groups + [index_group, stack_group]).sum()
df = df/1e6
df = df.drop('Social-housing', level='Occupancy status')
df = df / df.groupby(subplot_groups + [index_group]).sum()

df.to_csv(os.path.join(folder, 'stock_heater_ini.csv'))
make_stacked_bar_subplot(df, color=resources_data['colors'], subplot_groups=subplot_groups, index_group=index_group, stack_group=stack_group,
                         save=os.path.join(folder, 'stock_heater_ini.pdf'), format_y=lambda y, _: '{:.0%}'.format(y),
                         annotate=None)

In [ ]:
df = stock_end.groupby(subplot_groups + [index_group, stack_group]).sum()
df = df/1e6
df = df.drop('Social-housing', level='Occupancy status')
df_reference = df.copy()

In [ ]:
df = df / df.groupby(subplot_groups + [index_group]).sum()

df.to_csv(os.path.join(folder, 'stock_heater_end.csv'))

make_stacked_bar_subplot(df, color=resources_data['colors'], subplot_groups=subplot_groups, index_group=index_group, stack_group=stack_group,
                         save=os.path.join(folder, 'stock_heater_end_reference.png'), format_y=lambda y, _: '{:.0%}'.format(y),
                         annotate=None)

In [ ]:
stock_ban = pd.read_csv(os.path.join(folder, 'Ban', 'stock.csv'))
stock_ban.set_index(['Occupancy status', 'Income owner', 'Income tenant', 'Housing type', 'Heating system', 'Performance'], inplace=True)
stock_end_ban = stock_ban.iloc[:, -1]

In [ ]:
df = stock_end_ban.groupby(subplot_groups + [index_group, stack_group]).sum()
df = df/1e6
df = df.drop('Social-housing', level='Occupancy status')
df_ban = df.copy()

In [ ]:
df_diff = df_ban - df_reference
# df_diff = df_diff[abs(df_diff) >  0.01]
df_diff.to_csv(os.path.join(folder, 'stock_heater_diff_value.csv'))


df_diff = df_diff / df_reference.groupby(['Housing type', 'Occupancy status', 'Income tenant']).sum()
df_diff = df_diff[abs(df_diff) > 0.01]
df_diff.to_csv(os.path.join(folder, 'stock_heater_diff_ratio.csv'))
make_stacked_bar_subplot(df_diff, color=resources_data['colors'], subplot_groups=subplot_groups, index_group=index_group, stack_group=stack_group,
                         save=os.path.join(folder, 'stock_heater_difference.png'), format_y=lambda y, _: '{:.1%}'.format(y), 
                         annotate=None)


In [ ]:
df = df / df.groupby(subplot_groups + [index_group]).sum()
df.to_csv(os.path.join(folder, 'stock_heater_end_reference.csv'))
make_stacked_bar_subplot(df, color=resources_data['colors'], subplot_groups=subplot_groups, index_group=index_group, stack_group=stack_group,
                         save=os.path.join(folder, 'stock_heater_end_reference.png'), format_y=lambda y, _: '{:.0%}'.format(y),
                         annotate=None)